# Heritage Short: Wan 2.2 TI2V-5B moving shots

> **Status (2026-11-25): does not fit a free Kaggle T4.** See FAILURES.md F-26 and EXPERIMENTS.md EXP-12.
> - With the patches below, the model *loads*: peak RAM 22/31 GB.
> - It then runs out of GPU memory at denoising step 0. That's about 27k attention tokens, with no flash-attn and no bf16 on sm_75.
> - The film ships with Ken Burns moves (ARCHITECTURE.md §5).
>
> Run this notebook on a **≥ 24 GB GPU** (L4 / A10 / A100), where flash-attn and bf16 work. The job and import contract is unchanged, so clips slot in shot by shot.

**Run order:** cells 1 → 5. One clip at a time: two processes overflow host RAM.

In [ ]:
# Common setup: job files, output folder, helpers. Rerun this cell after any kernel restart.
import glob, json, os, subprocess, sys, time, datetime, shutil, re
from pathlib import Path
os.environ["MPLBACKEND"] = "Agg"   # subprocesses inherit the notebook's inline backend otherwise (F-25)

jobs = sorted(glob.glob("/kaggle/input/**/job.json", recursive=True))
def job(stage): return Path(next(j for j in jobs if json.load(open(j)).get("stage") == stage))
OUT = Path("/kaggle/working/kaggle_results")

def run(cmd, **kw):                 # stream a command; raise on failure
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)), flush=True)
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True, **kw)

def log(folder, stage, shot, model, params, seed, status, latency, output=None, error=None):
    rec = {"ts": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
           "run_id": "kaggle-" + stage, "stage": stage, "shot": shot, "provider": "kaggle", "model": model,
           "params": params, "seed": seed, "input_hash": None, "status": status, "attempt": 1,
           "latency_s": round(latency, 3), "gpu_s": round(latency, 3), "cost_usd_list": None,
           "output": output, "error": str(error)[-500:] if error else None}
    with open(folder / "runs_fragment.jsonl", "a") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")

run("nvidia-smi --query-gpu=index,name,memory.total --format=csv")
run(f"{sys.executable} -m pip install -q uv")

SHOTS_JOB = job("shots"); SHOTS = json.loads(SHOTS_JOB.read_text())
missing = [j["image"] for j in SHOTS["jobs"] if not (SHOTS_JOB.parent / j["image"]).exists()]
assert not missing, missing
(OUT / "shots").mkdir(parents=True, exist_ok=True)
print(len(SHOTS["jobs"]), "jobs", SHOTS["params"])

In [ ]:
# 2. Wan venv (torch locked by a constraints file, F-25) + weights (about 10 min)
WAN, WAN_PY, CKPT = "/kaggle/working/Wan2.2", "/tmp/wan/bin/python", "/tmp/Wan2.2-TI2V-5B"
run(f"test -d {WAN} || git clone -q https://github.com/Wan-Video/Wan2.2.git {WAN}")
run("uv venv --clear /tmp/wan --python 3.11")
Path("/tmp/wan_constraints.txt").write_text("torch==2.5.1+cu124\ntorchvision==0.20.1+cu124\ntorchaudio==2.5.1+cu124\nnumpy<2\n")
W = (f"uv pip install --python {WAN_PY} -c /tmp/wan_constraints.txt "
     f"--extra-index-url https://download.pytorch.org/whl/cu124 --index-strategy unsafe-best-match")
run(f"grep -viE '^(flash.?attn|torch|torchvision|torchaudio)' {WAN}/requirements.txt | tr -d '\\r' > /tmp/wan_req.txt")
run(f"{W} torch torchvision torchaudio -r /tmp/wan_req.txt huggingface_hub einops")
run([WAN_PY, "-c", f"from huggingface_hub import snapshot_download; snapshot_download('Wan-AI/Wan2.2-TI2V-5B', local_dir='{CKPT}')"])

In [ ]:
# 3. Patches for a GPU without flash-attn (F-26). Idempotent.
# a) Only ti2v is used: make the other modes' imports optional (they need decord, peft, ...)
init = Path(WAN) / "wan/__init__.py"; s = init.read_text()
if "# kaggle-optional" not in s:
    def opt(m):
        line = m.group(0)
        if "textimage2video" in line or "import configs" in line: return line
        return f"try:\n    {line}  # kaggle-optional\nexcept ImportError as e:\n    print('[wan] skipped:', e)"
    init.write_text(re.sub(r"^from \.\S+ import .+$", opt, s, flags=re.M))
# b) Load the DiT straight in its target dtype (fp32 first overflowed 31 GB host RAM)
f = Path(WAN) / "wan/textimage2video.py"; s = f.read_text()
if "kaggle-lowram" not in s:
    s2 = re.sub(r"WanModel\.from_pretrained\(\s*checkpoint_dir\s*\)",
                "WanModel.from_pretrained(checkpoint_dir, torch_dtype=config.param_dtype, low_cpu_mem_usage=True)  # kaggle-lowram", s)
    assert s2 != s, "from_pretrained pattern not found"
    f.write_text(s2)
# c) flash_attention -> fp16 memory-efficient SDPA
PATCH = '''

# kaggle-t4: fp16 memory-efficient SDPA (no flash-attn, no bf16 kernels on sm_75)
from torch.nn.attention import sdpa_kernel, SDPBackend
def flash_attention(q, k, v, q_lens=None, k_lens=None, dropout_p=0., softmax_scale=None, q_scale=None,
                    causal=False, window_size=(-1, -1), deterministic=False, dtype=torch.bfloat16, version=None):
    od = q.dtype
    if q_scale is not None: q = q * q_scale
    q, k, v = (t.transpose(1, 2).to(torch.float16) for t in (q, k, v))
    with sdpa_kernel([SDPBackend.EFFICIENT_ATTENTION]):
        out = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=softmax_scale)
    return out.transpose(1, 2).contiguous().to(od)
'''
att = Path(WAN) / "wan/modules/attention.py"
att.write_text(att.read_text().split("\n\n# kaggle-t4")[0] + PATCH)
run([WAN_PY, "-c", f"""
import sys, os; sys.path.insert(0, '{WAN}'); os.chdir('{WAN}')
import torch, einops
assert sys.prefix == '/tmp/wan' and torch.__version__.startswith('2.5.1'), (sys.prefix, torch.__version__)
import wan
from wan.modules import model as m
from wan.modules.attention import flash_attention
assert m.flash_attention is flash_attention
from wan.configs import SUPPORTED_SIZES; assert '1280*704' in SUPPORTED_SIZES['ti2v-5B']
q = torch.randn(1, 27280, 24, 128, device='cuda', dtype=torch.bfloat16); flash_attention(q, q, q); torch.cuda.synchronize()
print('WAN OK  torch', torch.__version__, torch.cuda.get_device_name(0), f'attn peak {{torch.cuda.max_memory_allocated()/1e9:.1f}} GB')
"""])

In [ ]:
# 4. Clips, one at a time (MAX_JOBS = 1 first, to measure)
MAX_JOBS, STEPS, TIMEOUT = 1, None, 5400
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
todo = [j for j in SHOTS["jobs"][:MAX_JOBS] if not (OUT / "shots" / j["out"]).exists()]
for i, j in enumerate(todo, 1):
    out = OUT / "shots" / j["out"]
    cmd = [WAN_PY, "generate.py", "--task", "ti2v-5B", "--size", SHOTS["params"]["size"], "--ckpt_dir", CKPT,
           "--offload_model", "True", "--convert_model_dtype", "--t5_cpu",
           "--image", str(SHOTS_JOB.parent / j["image"]), "--prompt", j["prompt"],
           "--base_seed", str(j["seed"]), "--frame_num", str(SHOTS["params"]["frame_num"]), "--save_file", str(out)]
    if STEPS: cmd += ["--sample_steps", str(STEPS)]
    params = {**SHOTS["params"], **({"sample_steps": STEPS} if STEPS else {})}
    print(f"[{i}/{len(todo)}] start {j['shot']}", flush=True); t0 = time.time()
    try:
        r = subprocess.run(cmd, cwd=WAN, env=env, capture_output=True, text=True, timeout=TIMEOUT)
        if r.returncode != 0 or not out.exists(): raise RuntimeError((r.stderr or r.stdout)[-1500:])
        log(OUT / "shots", "shots", j["shot"], SHOTS["model"], params, j["seed"], "ok", time.time() - t0, j["out"])
        print(f"[{i}/{len(todo)}] {j['shot']} ok  {(time.time()-t0)/60:.1f} min", flush=True)
    except Exception as e:
        out.unlink(missing_ok=True)
        log(OUT / "shots", "shots", j["shot"], SHOTS["model"], params, j["seed"], "error", time.time() - t0, None, e)
        print(f"[{i}/{len(todo)}] {j['shot']} FAILED ({(time.time()-t0)/60:.1f} min):\n{str(e)[-1500:]}", flush=True)

In [ ]:
# 5. Zip for download, then locally: py -m pipeline shots --import outputs/kaggle_results/shots
shutil.make_archive("/kaggle/working/shots_results", "zip", OUT / "shots")
print("Download /kaggle/working/shots_results.zip from the Output panel")